## EDA 

In [3]:
import pandas as pd
from pathlib import Path

proyecto_root = Path.cwd().parent
ruta_csv = proyecto_root / "data" / "df_modelo_para_tarifacion.csv"

df_eda = pd.read_csv(ruta_csv)
print(df_eda.shape)
df_eda.head()

(260853, 32)


,Afiliado_Id,FechaNacimiento_dt,edad,grupo_edad,CIUDAD_NORM,Sexo_Cd_limpio,CANCER,DIABETES,ENF_CARDIACA,HIPERTENSION,...,rec_CONSULTAS,rec_EXAMENES MEDICOS,rec_HOSPITALIZACIONES COMPLEJAS,rec_HOSPITALIZACIONES SIMPLES,rec_LABORATORIO,rec_TRATAMIENTO DE CANCER,en_exposicion,en_siniestros,categoria_cobertura,sin_exposicion_match
0,921437,1968-04-30,57,46-60,MEDELLIN,F,0,0,0,0,...,0,1,0,0,0,0,1,1,Completo (Exp + Sin),0
1,60504878,2012-02-18,14,0-17,MEDELLIN,M,0,0,0,0,...,0,0,0,1,0,0,1,1,Completo (Exp + Sin),0
2,55074222,2014-10-23,11,0-17,MEDELLIN,F,0,0,0,0,...,0,4,1,1,0,0,1,1,Completo (Exp + Sin),0
3,23690690,1989-06-27,36,31-45,CARTAGENA,F,0,0,0,0,...,0,4,0,0,0,0,1,1,Completo (Exp + Sin),0
4,45506882,2009-06-30,16,0-17,CALI,M,0,0,0,0,...,0,1,0,1,0,0,1,1,Completo (Exp + Sin),0


In [4]:
# 1) Número total de variables
print("N° de variables:", len(df_eda.columns))

# 2) Lista completa de nombres
print(df_eda.columns.tolist())

# 3) Tabla ordenada de variables (más legible)
display(
    pd.DataFrame({
        "variable": df_eda.columns,
        "dtype": df_eda.dtypes.astype(str).values,
        "nulos": df_eda.isna().sum().values
    }).sort_values("variable").reset_index(drop=True)
)

N° de variables: 32
['Afiliado_Id', 'FechaNacimiento_dt', 'edad', 'grupo_edad', 'CIUDAD_NORM', 'Sexo_Cd_limpio', 'CANCER', 'DIABETES', 'ENF_CARDIACA', 'HIPERTENSION', 'ENF_PULMONAR', 'num_condiciones', 'dias_expuesto_total', 'meses_expuesto_total', 'n_polizas', 'n_reclamaciones', 'n_eventos', 'total_pagado', 'fecha_primera_reclamacion', 'fecha_ultima_reclamacion', 'rec_CIRUGIA', 'rec_CONSULTA MEDICA', 'rec_CONSULTAS', 'rec_EXAMENES MEDICOS', 'rec_HOSPITALIZACIONES COMPLEJAS', 'rec_HOSPITALIZACIONES SIMPLES', 'rec_LABORATORIO', 'rec_TRATAMIENTO DE CANCER', 'en_exposicion', 'en_siniestros', 'categoria_cobertura', 'sin_exposicion_match']


,variable,dtype,nulos
0,Afiliado_Id,int64,0
1,CANCER,int64,0
2,CIUDAD_NORM,object,0
3,DIABETES,int64,0
4,ENF_CARDIACA,int64,0
5,ENF_PULMONAR,int64,0
6,FechaNacimiento_dt,object,0
7,HIPERTENSION,int64,0
8,Sexo_Cd_limpio,object,0
9,categoria_cobertura,object,0


## EDA (Análisis exploratorio de datos) Formulación de preguntas claves para su desarrollo

1. ¿Quién se enferma más? Edad, sexo, condiciones preexistentes
2. ¿Cuánto cuesta enfermarse?  para responderla tenemos en cuenta; total_pagado, n_reclamaciones, n_eventos
3. ¿Qué tan seguido?  Es basicamente la  Frecuencia de uso del seguro
4. ¿Dónde?  En que Ciudad
5. ¿De qué? Tipos de reclamación (La respondemos con las variables rec_)

In [6]:
# ── RESUMEN EJECUTIVO ─────────────────────────────────────────────────────────

total_asegurados   = len(df_eda)
usaron_seguro      = (df_eda['n_reclamaciones'] > 0).sum()
tasa_uso           = usaron_seguro / total_asegurados * 100
costo_total        = df_eda['total_pagado'].sum()
costo_promedio     = df_eda[df_eda['n_reclamaciones'] > 0]['total_pagado'].mean()
costo_mediano      = df_eda[df_eda['n_reclamaciones'] > 0]['total_pagado'].median()
exposicion_prom    = df_eda['meses_expuesto_total'].mean()
edad_promedio      = df_eda['edad'].mean()
recl_promedio      = df_eda[df_eda['n_reclamaciones'] > 0]['n_reclamaciones'].mean()

print('=' * 65)
print('         RESUMEN EJECUTIVO — UdeA Insurance')
print('=' * 65)
print(f'  Total asegurados analizados:       {total_asegurados:>12,}')
print(f'  Usaron el seguro (≥1 reclamación): {usaron_seguro:>12,}  ({tasa_uso:.1f}%)')
print(f'  No usaron el seguro:               {total_asegurados - usaron_seguro:>12,}  ({100 - tasa_uso:.1f}%)')
print('-' * 65)
print(f'  Costo total pagado:                ${costo_total:>18,.0f}')
print(f'  Costo promedio (quienes usaron):   ${costo_promedio:>18,.0f}')
print(f'  Costo mediano  (quienes usaron):   ${costo_mediano:>18,.0f}')
print(f'  Reclamaciones promedio por usuario:{recl_promedio:>15.1f}')
print('-' * 65)
print(f'  Exposición promedio:               {exposicion_prom:>14.1f} meses')
print(f'  Edad promedio de la cartera:       {edad_promedio:>14.1f} años')
print('=' * 65)

# La brecha entre promedio y mediana del costo dice todo:
# si promedio >> mediana, hay una minoría que concentra costos altísimos.
print(f'\nBrecha promedio/mediana: x{costo_promedio/costo_mediano:.1f}')
print('→ Si es > 2x, la distribución de costos es muy asimétrica (normal en salud)')

         RESUMEN EJECUTIVO — UdeA Insurance
  Total asegurados analizados:            260,853
  Usaron el seguro (≥1 reclamación):      219,800  (84.3%)
  No usaron el seguro:                     41,053  (15.7%)
-----------------------------------------------------------------
  Costo total pagado:                $ 1,194,478,769,631
  Costo promedio (quienes usaron):   $         5,434,389
  Costo mediano  (quienes usaron):   $         1,512,496
  Reclamaciones promedio por usuario:            8.3
-----------------------------------------------------------------
  Exposición promedio:                         11.2 meses
  Edad promedio de la cartera:                 35.6 años

Brecha promedio/mediana: x3.6
→ Si es > 2x, la distribución de costos es muy asimétrica (normal en salud)
